```{contents}
```
## Generalization


**Generalization** is the ability of a trained neural network to perform well on **unseen data** drawn from the same distribution as the training set.
A model that fits training data perfectly but fails on new data has **poor generalization** (overfitting).
A model that performs well on both training and unseen data has **good generalization**.

---

### Intuition

A deep network does not memorize the dataset; it **learns a function** that captures the **underlying structure** of the data.

Think of training as fitting a surface in high-dimensional space:

* **Underfitting** → surface too simple
* **Overfitting** → surface too complex, bends around noise
* **Good generalization** → surface captures only true signal

---

### Bias–Variance Tradeoff

| Concept  | Meaning                                            |
| -------- | -------------------------------------------------- |
| Bias     | Error from overly simplistic assumptions           |
| Variance | Error from sensitivity to data noise               |
| Goal     | Minimize total error: **Bias² + Variance + Noise** |

Generalization is achieved by **balancing bias and variance**.

---

### Generalization Workflow

**Data → Model → Training → Validation → Regularization → Test**

1. **Split dataset**

   * Train
   * Validation
   * Test
2. **Train model** on training set
3. **Tune hyperparameters** on validation set
4. **Apply regularization** if overfitting occurs
5. **Evaluate final model** on test set

---

### Failure Modes

| Symptom                                   | Diagnosis      |
| ----------------------------------------- | -------------- |
| High training accuracy, low test accuracy | Overfitting    |
| Low training and test accuracy            | Underfitting   |
| High variance across runs                 | Model unstable |

---

### Core Techniques for Better Generalization

| Method                 | Effect                           |
| ---------------------- | -------------------------------- |
| Data augmentation      | Increases effective dataset size |
| Weight decay (L2)      | Penalizes complex models         |
| Dropout                | Prevents co-adaptation           |
| Early stopping         | Stops before overfitting         |
| Batch normalization    | Stabilizes optimization          |
| Model capacity control | Limits overfitting               |

---

### Mathematical View

Training minimizes **empirical risk**:

[
\min_\theta \frac{1}{N} \sum_{i=1}^{N} L(f_\theta(x_i), y_i)
]

Generalization depends on how well this approximates the **expected risk** over the true data distribution.

---

### PyTorch Demonstration

#### Dataset

```python
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import torch, torch.nn as nn
import torch.optim as optim
```

```python
X, y = make_moons(2000, noise=0.2)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)
```

#### Overfitting Model

```python
class OverfitNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.net(x)
```

#### Regularized Model

```python
class GeneralizedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        return self.net(x)
```

#### Training Loop

```python
def train(model):
    optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(200):
        model.train()
        loss = loss_fn(model(X_train), y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 50 == 0:
            model.eval()
            val_loss = loss_fn(model(X_val), y_val)
            print(epoch, loss.item(), val_loss.item())
```

---

### Expected Behavior

| Model          | Training Loss   | Validation Loss |
| -------------- | --------------- | --------------- |
| OverfitNet     | Very low        | Increasing      |
| GeneralizedNet | Slightly higher | Lower & stable  |

---

### Practical Generalization Checklist

* Use validation sets for tuning
* Track training vs validation curves
* Reduce capacity when overfitting
* Apply regularization consistently
* Prefer simpler models when performance matches

---

### Variants of Generalization Strategies

| Strategy                  | Use Case                    |
| ------------------------- | --------------------------- |
| Structural regularization | Model pruning, smaller nets |
| Stochastic regularization | Dropout, noise injection    |
| Data-based                | Augmentation, mixup         |
| Optimization-based        | Early stopping, SGD noise   |
| Bayesian methods          | Uncertainty-aware models    |

---

### Summary

Generalization is the central goal of deep learning.
All architectural choices, training procedures, and regularization methods exist to improve the model’s ability to **learn structure rather than memorize data**.
